# 04 · Corner Case Safety Monitor

真实智驾系统通常不会把所有安全责任交给一个端到端模型。一个独立的 safety monitor 可以读取候选轨迹、目标状态、置信度和传感器健康度，在 corner case 下触发减速、最小风险动作或交给更保守的 planner。

本 notebook 用合成数据练习一个最小的 rule-based monitor：

- longitudinal gap 与 time-to-collision（TTC）计算；
- perception confidence 与 sensor age 的保守门控；
- 事件级 recall、false-alarm rate 和阈值扫描；
- 交互修改阈值，观察“高召回”和“少误报”的冲突。

这不是安全认证算法，也不能替代真实车辆上的冗余、形式化验证、仿真和道路测试。它的学习价值在于：把 corner-case 假设、保护条件、可观测指标和 fallback 行为写成可复现实验。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
from ipywidgets import interact, FloatSlider

plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True


## Part A — 从相对运动得到一个可解释的风险信号

在一条简化车道上，ego 车以固定速度前进，前方障碍物可以静止、低速行驶或横向偏离。定义：

- gap = obstacle_x - ego_x - bumper_margin
- TTC = gap / max(ego_speed - obstacle_speed, epsilon)，只在 closing speed > 0 且目标在本车道时有效
- fallback = TTC 小于阈值，或 clearance 小于硬阈值，或置信度 / 传感器健康度不足

TTC 是很有用的第一层信号，但它依赖目标跟踪、坐标系和时间同步；corner case 练习的重点正是验证这些输入变坏时 monitor 是否仍然保守。


In [ ]:
def build_scenario(
    obstacle_speed=1.5,
    lateral_offset=0.0,
    confidence_dip=0.25,
    sensor_age_spike=0.0,
    obstacle_start=42.0,
    ego_speed=8.0,
    duration=12.0,
    dt=0.1,
):
    time = np.arange(0.0, duration, dt)
    ego_x = ego_speed * time
    obstacle_x = obstacle_start + obstacle_speed * time
    gap = obstacle_x - ego_x - 4.0
    closing_speed = ego_speed - obstacle_speed
    same_lane = abs(lateral_offset) <= 1.4
    ttc = np.where(
        same_lane & (closing_speed > 0.0),
        gap / max(closing_speed, 1e-3),
        np.inf,
    )
    clearance = np.sqrt(np.maximum(gap, 0.0) ** 2 + lateral_offset ** 2)
    dip_shape = np.exp(-0.5 * ((time - 6.0) / 0.8) ** 2)
    confidence = np.clip(0.98 - confidence_dip * dip_shape, 0.0, 1.0)
    sensor_age = 0.05 + sensor_age_spike * dip_shape
    return {
        'time': time,
        'ego_x': ego_x,
        'obstacle_x': obstacle_x,
        'lateral_offset': lateral_offset,
        'gap': gap,
        'clearance': clearance,
        'ttc': ttc,
        'confidence': confidence,
        'sensor_age': sensor_age,
        'same_lane': same_lane,
    }

def safety_monitor(
    scenario,
    ttc_threshold=2.0,
    confidence_floor=0.60,
    max_sensor_age=0.25,
    hard_clearance=1.0,
):
    ttc_risk = scenario['ttc'] <= ttc_threshold
    clearance_risk = scenario['clearance'] <= hard_clearance
    confidence_risk = scenario['confidence'] <= confidence_floor
    sensor_health_risk = scenario['sensor_age'] >= max_sensor_age
    trigger = ttc_risk | clearance_risk | confidence_risk | sensor_health_risk
    return {
        'trigger': trigger,
        'ttc_risk': ttc_risk,
        'clearance_risk': clearance_risk,
        'confidence_risk': confidence_risk,
        'sensor_health_risk': sensor_health_risk,
    }


In [ ]:
scenario = build_scenario(
    obstacle_speed=1.5,
    lateral_offset=0.0,
    confidence_dip=0.25,
    sensor_age_spike=0.0,
)
decision = safety_monitor(scenario)

fallback_time = scenario['time'][decision['trigger']]
print(f"same lane: {scenario['same_lane']}")
print(f"minimum gap: {scenario['gap'].min():.2f} m")
print(f"minimum finite TTC: {np.nanmin(np.where(np.isfinite(scenario['ttc']), scenario['ttc'], np.nan)):.2f} s")
print(f"fallback starts at: {fallback_time[0]:.2f} s" if len(fallback_time) else 'no fallback trigger')

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(scenario['ego_x'], np.zeros_like(scenario['ego_x']), label='ego')
ax[0].plot(scenario['obstacle_x'], np.full_like(scenario['obstacle_x'], scenario['lateral_offset']), label='obstacle')
ax[0].set_title('longitudinal scene')
ax[0].set_xlabel('x / m')
ax[0].set_ylabel('lateral / m')
ax[0].legend()

ax[1].plot(scenario['time'], scenario['gap'], label='gap')
ax[1].plot(scenario['time'], scenario['ttc'], label='TTC')
ax[1].axhline(2.0, color='tab:red', linestyle='--', label='TTC threshold')
ax[1].set_ylim(-2, 8)
ax[1].set_title('risk signals')
ax[1].set_xlabel('time / s')
ax[1].legend()

ax[2].plot(scenario['time'], scenario['confidence'], label='confidence')
ax[2].plot(scenario['time'], scenario['sensor_age'], label='sensor age / s')
ax[2].axhline(0.60, color='tab:orange', linestyle='--', label='confidence floor')
ax[2].axhline(0.25, color='tab:red', linestyle=':', label='age limit')
ax[2].fill_between(scenario['time'], 0, decision['trigger'].astype(float), alpha=0.2, label='fallback')
ax[2].set_ylim(0, 1.1)
ax[2].set_title('health and decision')
ax[2].set_xlabel('time / s')
ax[2].legend()
plt.tight_layout()


### 交互练习 1：阈值不是越激进越好

下面的滑块会重新画出同一个场景。观察：

- TTC 阈值变大时，fallback 是否更早，但误报风险也更高？
- confidence floor 变大时，模型的“低置信度”是否会把大量正常场景挡住？
- 横向偏移超过本车道范围后，为什么 TTC 不能直接作为同样的风险信号？
- sensor age spike 出现时，是否应该直接 fallback，还是降低速度并等待下一帧？


In [ ]:
def show_monitor(
    ttc_threshold=2.0,
    confidence_floor=0.60,
    lateral_offset=0.0,
    confidence_dip=0.25,
    sensor_age_spike=0.0,
):
    scenario = build_scenario(
        lateral_offset=lateral_offset,
        confidence_dip=confidence_dip,
        sensor_age_spike=sensor_age_spike,
    )
    decision = safety_monitor(
        scenario,
        ttc_threshold=ttc_threshold,
        confidence_floor=confidence_floor,
    )
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(scenario['time'], scenario['gap'], label='gap / m')
    ax[0].plot(scenario['time'], scenario['ttc'], label='TTC / s')
    ax[0].axhline(ttc_threshold, color='tab:red', linestyle='--', label='TTC threshold')
    ax[0].set_ylim(-2, 8)
    ax[0].set_xlabel('time / s')
    ax[0].set_title(f"same_lane={scenario['same_lane']}, fallback_frames={decision['trigger'].sum()}")
    ax[0].legend()

    ax[1].plot(scenario['time'], scenario['confidence'], label='confidence')
    ax[1].plot(scenario['time'], scenario['sensor_age'], label='sensor age')
    ax[1].axhline(confidence_floor, color='tab:orange', linestyle='--', label='confidence floor')
    ax[1].axhline(0.25, color='tab:red', linestyle=':', label='age limit')
    ax[1].fill_between(scenario['time'], 0, decision['trigger'].astype(float), alpha=0.2, label='fallback')
    ax[1].set_ylim(0, 1.1)
    ax[1].set_xlabel('time / s')
    ax[1].set_title('monitor output')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(
    show_monitor,
    ttc_threshold=FloatSlider(min=0.5, max=5.0, step=0.25, value=2.0, description='TTC'),
    confidence_floor=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.60, description='conf'),
    lateral_offset=FloatSlider(min=-2.5, max=2.5, step=0.25, value=0.0, description='lateral'),
    confidence_dip=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.25, description='conf dip'),
    sensor_age_spike=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.0, description='age spike'),
);


## Part B — 事件级评估：高 recall 与低 false alarm 的冲突

逐帧 accuracy 往往会被大量安全帧“冲高”，不适合描述 safety monitor。这里改用 episode-level 判断：

- ground truth critical：目标在本车道，且整个片段内 clearance 曾低于 2 m；
- predicted critical：monitor 至少触发过一次；
- recall = TP / (TP + FN)，优先保证真正危险事件被覆盖；
- false-alarm rate = FP / (FP + TN)，衡量正常片段被不必要接管的程度。

这是一个简化评估，但它已经比只报一个 frame accuracy 更接近智驾安全模块的思路。


In [ ]:
def evaluate_batch(
    n=500,
    ttc_threshold=2.0,
    confidence_floor=0.60,
    max_sensor_age=0.25,
    seed=123,
):
    local_rng = np.random.default_rng(seed)
    truth = []
    prediction = []
    for _ in range(n):
        offset = local_rng.uniform(-2.5, 2.5)
        scenario = build_scenario(
            obstacle_speed=local_rng.uniform(0.0, 7.0),
            lateral_offset=offset,
            confidence_dip=local_rng.uniform(0.0, 0.6),
            sensor_age_spike=local_rng.uniform(0.0, 0.45),
            obstacle_start=local_rng.uniform(30.0, 58.0),
        )
        decision = safety_monitor(
            scenario,
            ttc_threshold=ttc_threshold,
            confidence_floor=confidence_floor,
            max_sensor_age=max_sensor_age,
        )
        is_critical = bool(
            scenario['same_lane'] and np.min(scenario['clearance']) < 2.0
        )
        truth.append(is_critical)
        prediction.append(bool(decision['trigger'].any()))

    truth = np.asarray(truth, dtype=bool)
    prediction = np.asarray(prediction, dtype=bool)
    tp = np.sum(truth & prediction)
    fn = np.sum(truth & ~prediction)
    fp = np.sum(~truth & prediction)
    tn = np.sum(~truth & ~prediction)
    recall = tp / max(tp + fn, 1)
    false_alarm_rate = fp / max(fp + tn, 1)
    return {
        'tp': int(tp),
        'fn': int(fn),
        'fp': int(fp),
        'tn': int(tn),
        'recall': recall,
        'false_alarm_rate': false_alarm_rate,
    }

metrics = evaluate_batch()
print(metrics)


In [ ]:
thresholds = np.linspace(0.5, 5.0, 19)
recalls = []
false_alarm_rates = []
for threshold in thresholds:
    result = evaluate_batch(ttc_threshold=threshold)
    recalls.append(result['recall'])
    false_alarm_rates.append(result['false_alarm_rate'])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(false_alarm_rates, recalls, marker='o')
for x_value, y_value, threshold in zip(false_alarm_rates, recalls, thresholds):
    if threshold in {0.5, 2.0, 5.0}:
        ax.annotate(f'TTC={threshold:.1f}', (x_value, y_value))
ax.set_xlabel('false-alarm rate')
ax.set_ylabel('event recall')
ax.set_title('TTC threshold sweep')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.show()


### 交互练习 2：加入“保护策略”而不是只改阈值

请尝试把 monitor 变成更像一个可审计的 safety layer：

1. 扫描 TTC threshold 和 confidence floor，寻找 recall ≥ 0.99 的最低 false-alarm 区域。
2. 加入 hysteresis：触发阈值和解除阈值不同，避免 fallback 在边界附近抖动。
3. 把 sensor age 从单阈值改成“短时降速、持续失联才最小风险停车”。
4. 把硬规则输出记录为 reason code，例如 TTC_RISK、LOW_CONFIDENCE、STALE_SENSOR，并统计每类触发比例。
5. 如果后面加入小模型，必须在规则 monitor 外保留 hard collision check，并在独立 corner-case 集上验证，不要只看平均验证集指标。


In [ ]:
def benchmark_monitor(repeats=1000):
    scenario = build_scenario()
    latencies_us = []
    for _ in range(repeats):
        start = perf_counter()
        _ = safety_monitor(scenario)
        latencies_us.append((perf_counter() - start) * 1e6)
    latencies_us = np.asarray(latencies_us)
    print(f'safety_monitor p50: {np.percentile(latencies_us, 50):.2f} us')
    print(f'safety_monitor p95: {np.percentile(latencies_us, 95):.2f} us')
    print(f'safety_monitor p99: {np.percentile(latencies_us, 99):.2f} us')

benchmark_monitor()


## 完成标准

完成本 notebook 后，请保留一份实验记录，至少包括：

- 一张可视化：场景、TTC / gap、置信度 / sensor age 和 fallback 时间；
- 一个阈值扫描图，报告 episode-level recall 与 false-alarm rate；
- 一个失败案例：monitor 漏掉了什么，或者为什么触发了不必要的 fallback；
- 至少一个 reason code 和一个 hysteresis / degraded-mode 设计；
- p50 / p95 / p99 时延，以及你认为真实部署还缺少哪些验证：时钟同步、坐标标定、目标遮挡、传感器失效、仿真到现实偏差等。
